# Offline Security Evaluation (Colab)

Bu notebook, daha önceden koşturulmuş ve elde edilmiş `round_X.pkl` snapshot'ları üzerinde çevrimdışı (offline) olarak güvenlik saldırılarını değerlendirir.
**Çalışma zamanı:** Üst menüden **Runtime → Change runtime type → GPU** (önerilir).

## 1) Projeyi getir

In [ ]:
import os
import subprocess

CONTENT = "/content"
REPO_URL = "https://github.com/mervegulec2/privproject.git"
BRANCH = "main"
PROJECT_ROOT = os.path.join(CONTENT, "privproject")

os.chdir(CONTENT)
if not os.path.isdir(PROJECT_ROOT):
    subprocess.check_call(["git", "clone", REPO_URL, "privproject"])
    os.chdir(PROJECT_ROOT)
    subprocess.check_call(["git", "checkout", BRANCH])
else:
    os.chdir(PROJECT_ROOT)
    subprocess.check_call(["git", "pull"])

os.environ["PROJECT_ROOT"] = os.getcwd()
print("PROJECT_ROOT:", os.environ["PROJECT_ROOT"])

## 2) ZIP Yedeklerini Geri Yükle (Eğer Drive'dan kullanıyorsan)

Bir önceki notebook'ta oluşan `outputs` klasörünü Colab ortamına atmadıysan, kaydettiğin ZIP dosyasını yüklemelisin. Eğer aynı oturumdaysan ve dosyalar zaten `outputs/security/snapshots/` içindeyse bu bloğu geçebilirsin.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/baseline_outputs.zip -d /content/privproject/

## 3) Bağımlılıklar

In [ ]:
import os
import subprocess
import sys

root = os.environ.get("PROJECT_ROOT", "/content/privproject")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=root,
)
# dotenv eksikse ayrıca kuralım ki .env dosyamızı kesinlikle okusun
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "python-dotenv"],
    cwd=root,
)
print("pip OK:", root)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "flwr[simulation]",
    "ray[default]"
])

print("Done — now re-run the baseline cell, no restart needed this time.")

## 4) RECONSTRUCTION ATTACK (Round 1)

In [ ]:
!python eval_security.py --snapshot outputs/security/snapshots/round_1.pkl --attack reconstruction

## 5) MEMBERSHIP INFERENCE ATTACK (MIA) (Round 2)

In [ ]:
!python eval_security.py --snapshot outputs/security/snapshots/round_2.pkl --attack mia

## 6) CORRELATION PROPERTY ATTACK (CPA) (Round 1)

In [ ]:
!python eval_security.py --snapshot outputs/security/snapshots/round_1.pkl --attack cpa